In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
import tensorflow.keras.layers as tfl
from tensorflow.keras.models import Sequential, Model

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

## Loading Data

In [2]:
train = pd.read_csv("data/train_clean.csv")
test = pd.read_csv("data/test_clean.csv")
test_original = pd.read_csv("data/test.csv")
id = test_original["id"]

In [3]:
train_text = train["text"]
test_text = test["text"]
target = train["target"]

In [4]:
train_text.isna().sum(), test_text.isna().sum()

(0, 0)

No `none` in the data.

In [5]:
train_text

0            deeds reason earthquake may allah forgive us
1                   forest fire near la ronge sask canada
2       residents asked 'shelter place' notified offic...
3       13000 people receive wildfires evacuation orde...
4       got sent photo ruby alaska smoke wildfires pou...
                              ...                        
7607    two giant cranes holding bridge collapse nearb...
7608    control wild fires california even northern pa...
7609                    m194 [0104 utc]5km volcano hawaii
7610    police investigating ebike collided car little...
7611    latest homes razed northern california wildfir...
Name: text, Length: 7612, dtype: object

## Models

### Making Data Ready

In [6]:
X_train, X_test, y_train, y_test = train_test_split(train_text, target, test_size=0.1, random_state=42)

In [7]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def calculate_results(y_true, y_pred):
  """
  Calculates model accuracy, precision, recall and f1 score of a binary classification model.

  Args:
  -----
  y_true = true labels in the form of a 1D array
  y_pred = predicted labels in the form of a 1D array

  Returns a dictionary of accuracy, precision, recall, f1-score.
  """
  # Calculate model accuracy
  model_accuracy = accuracy_score(y_true, y_pred) * 100
  # Calculate model precision, recall and f1 score using "weighted" average
  model_precision, model_recall, model_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted")
  model_results = {"accuracy": model_accuracy,
                  "precision": model_precision,
                  "recall": model_recall,
                  "f1": model_f1}
  return model_results

### Naive Bayes (The Base Model)

In [8]:
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("nb", MultinomialNB())
])
pipe.fit(X_train, y_train)

Pipeline(steps=[('tfidf', TfidfVectorizer(stop_words='english')),
                ('nb', MultinomialNB())])

In [9]:
pipe.score(X_train, y_train), pipe.score(X_test, y_test)

(0.8982481751824818, 0.7913385826771654)

#### Base Model Predictions

In [10]:
pred = pipe.predict(test_text)
pred

array([1, 0, 1, ..., 1, 1, 1], dtype=int64)

In [11]:
base_model_performance = calculate_results(y_test, pipe.predict(X_test))
base_model_performance

{'accuracy': 79.13385826771653,
 'precision': 0.7974211556888723,
 'recall': 0.7913385826771654,
 'f1': 0.7865535576835297}

In [12]:
sub = pd.DataFrame({"id": id, "target": pred})
sub

,id,target
0,0,1
1,2,0
2,3,1
3,9,1
4,11,1
...,...,...
3258,10861,1
3259,10865,1
3260,10868,1
3261,10874,1


In [18]:
sub.to_csv(r"submissions\base_submission.csv", index=False)

In [19]:
!kaggle competitions submit -c nlp-getting-started -f submissions/base_submission.csv -m "Base Model"

Successfully submitted to Natural Language Processing with Disaster Tweets


  0%|          | 0.00/25.4k [00:00<?, ?B/s]
 31%|███▏      | 8.00k/25.4k [00:00<00:00, 73.9kB/s]
100%|██████████| 25.4k/25.4k [00:05<00:00, 4.43kB/s]


### DL Models

#### Tokenizing Data

In [14]:
MAX_VOCAB_SIZE = 10000
MAX_LENGTH = 20
tokenizer = tfl.TextVectorization(max_tokens=MAX_VOCAB_SIZE, output_sequence_length=MAX_LENGTH, name="tokenizer1")
tokenizer.adapt(train_text)

In [15]:
vocab = tokenizer.get_vocabulary()
vocab[:10], vocab[-10:]

(['', '[UNK]', 'like', 'e', 'amp', 'im', 'a', 'fire', 't', 'get'],
 ['nailreal',
  'nagaski',
  'naemolgo',
  'nades',
  'naayf',
  'naaa',
  'na',
  'n36',
  'n15b',
  'myths'])

#### Embedding Data

In [16]:
embedd = tfl.Embedding(input_dim=MAX_VOCAB_SIZE, output_dim=128, input_length=MAX_LENGTH, name="embedd_1")

#### Model 1

In [17]:
input = tfl.Input(shape=(1,), dtype="string", name="input1")
x = tokenizer(input)
x = embedd(x)
x = tfl.Flatten(name="flatten1")(x) 
x = tfl.Dense(128, activation="relu", name="dense1")(x)
output = tfl.Dense(1, activation="sigmoid")(x)
model = Model(inputs=input, outputs=output)
model.compile(optimizer="adam", loss="mse")
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input1 (InputLayer)         [(None, 1)]               0         
                                                                 
 tokenizer1 (TextVectorizati  (None, 20)               0         
 on)                                                             
                                                                 
 embedd_1 (Embedding)        (None, 20, 128)           1280000   
                                                                 
 flatten1 (Flatten)          (None, 2560)              0         
                                                                 
 dense1 (Dense)              (None, 128)               327808    
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                             